# Gold Layer: Borough Summary

Aggregates Silver listing data to produce one row per neighbourhood × city.
Computes pricing, occupancy, revenue, and review metrics, then scores each neighbourhood
for three investor personas: Yield Maximiser, Occupancy Optimiser, Quality Host.

Reads from `AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED` (multi-city).
Persona scores are normalised **per city** so cities are ranked independently.

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DB         = 'AIRBNB_INVESTMENT_DB'
SCHEMA_SIL = 'SILVER'
SCHEMA_GLD = 'GOLD'
TABLE_OUT  = 'BOROUGH_SUMMARY'

In [ ]:
def minmax_norm(series, invert=False):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    normed = (series - min_val) / (max_val - min_val)
    return 1 - normed if invert else normed


def load_silver(session):
    df = session.sql("""
        SELECT
            LISTING_ID,
            NEIGHBOURHOOD,
            PROPERTY_TYPE,
            ROOM_TYPE,
            PRICE,
            AVAILABILITY_365,
            ESTIMATED_OCCUPANCY_L365D,
            ESTIMATED_REVENUE_L365D,
            NUMBER_OF_REVIEWS,
            REVIEW_SCORES_RATING,
            REVIEW_SCORES_CLEANLINESS,
            REVIEW_SCORES_LOCATION,
            REVIEW_SCORES_CHECKIN,
            REVIEW_SCORES_COMMUNICATION,
            REVIEW_SCORES_VALUE,
            HOST_RESPONSE_RATE_PCT,
            HOST_ACCEPTANCE_RATE_PCT,
            HOST_IS_SUPERHOST,
            _FILENAME
        FROM AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED
    """).to_pandas()

    df['city'] = df['_FILENAME'].str.extract(
        r'raw/inside_airbnb/([^/]+)/')[0].str.upper()

    print(f'Loaded {len(df):,} listings')
    print(df.groupby('city').size())
    return df


def load_geo_data(session):
    df = session.sql("""
        SELECT NEIGHBOURHOOD, AREA_SQKM, _FILENAME
        FROM AIRBNB_INVESTMENT_DB.SILVER.NEIGHBOURHOOD_GEO_CLEANED
        WHERE AREA_SQKM IS NOT NULL AND AREA_SQKM > 0
    """).to_pandas()

    df['city'] = df['_FILENAME'].str.extract(
        r'raw/inside_airbnb/([^/]+)/')[0].str.upper()

    print(f'Loaded geo data for {len(df)} neighbourhoods')
    print(df.groupby('city').size())
    return df


def load_poi_data(session):
    df_poi = session.sql("""
        SELECT
            NEIGHBOURHOOD,
            AMENITY_GROUP,
            COUNT(*) AS poi_count
        FROM AIRBNB_INVESTMENT_DB.SILVER.POI_CLEANED
        WHERE AMENITY_GROUP IN (
            'Transport',
            'Attractions & Culture',
            'Dining & Nightlife'
        )
        GROUP BY NEIGHBOURHOOD, AMENITY_GROUP
    """).to_pandas()

    df_pivot = df_poi.pivot_table(
        index='NEIGHBOURHOOD',
        columns='AMENITY_GROUP',
        values='poi_count',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    df_pivot.columns.name = None
    df_pivot = df_pivot.rename(columns={
        'Transport':             'poi_transport',
        'Attractions & Culture': 'poi_attractions',
        'Dining & Nightlife':    'poi_dining',
    })

    for col in ['poi_transport', 'poi_attractions', 'poi_dining']:
        if col not in df_pivot.columns:
            df_pivot[col] = 0

    print(f'Loaded POI data for {len(df_pivot)} neighbourhoods')
    return df_pivot


def compute_metrics(df):
    df = df.copy()
    df['is_entire_home']    = (df['ROOM_TYPE'] == 'Entire home/apt').astype(int)
    df['is_private_room']   = (df['ROOM_TYPE'] == 'Private room').astype(int)
    df['is_shared_room']    = (df['ROOM_TYPE'] == 'Shared room').astype(int)
    df['is_hotel_room']     = (df['ROOM_TYPE'] == 'Hotel room').astype(int)
    df['is_superhost_flag'] = df['HOST_IS_SUPERHOST'].astype(float)

    df_metrics = df.groupby(['city', 'NEIGHBOURHOOD']).agg(
        listing_count            = ('LISTING_ID', 'count'),
        total_reviews            = ('NUMBER_OF_REVIEWS', 'sum'),
        avg_price                = ('PRICE', 'mean'),
        min_price                = ('PRICE', 'min'),
        max_price                = ('PRICE', 'max'),
        avg_occupancy            = ('ESTIMATED_OCCUPANCY_L365D', 'mean'),
        avg_revenue              = ('ESTIMATED_REVENUE_L365D', 'mean'),
        total_revenue            = ('ESTIMATED_REVENUE_L365D', 'sum'),
        avg_review_rating        = ('REVIEW_SCORES_RATING', 'mean'),
        avg_cleanliness_score    = ('REVIEW_SCORES_CLEANLINESS', 'mean'),
        avg_location_score       = ('REVIEW_SCORES_LOCATION', 'mean'),
        avg_checkin_score        = ('REVIEW_SCORES_CHECKIN', 'mean'),
        avg_communication_score  = ('REVIEW_SCORES_COMMUNICATION', 'mean'),
        avg_value_score          = ('REVIEW_SCORES_VALUE', 'mean'),
        room_type_variety        = ('ROOM_TYPE', 'nunique'),
        property_type_variety    = ('PROPERTY_TYPE', 'nunique'),
        count_entire_home_apt    = ('is_entire_home', 'sum'),
        count_private_room       = ('is_private_room', 'sum'),
        count_shared_room        = ('is_shared_room', 'sum'),
        count_hotel_room         = ('is_hotel_room', 'sum'),
        avg_host_response_rate   = ('HOST_RESPONSE_RATE_PCT', 'mean'),
        avg_host_acceptance_rate = ('HOST_ACCEPTANCE_RATE_PCT', 'mean'),
        pct_superhost            = ('is_superhost_flag',
                                    lambda x: x.mean() * 100),
    ).reset_index()

    df_metrics = df_metrics.rename(
        columns={'NEIGHBOURHOOD': 'neighbourhood_cleansed'}
    )

    float_cols = df_metrics.select_dtypes(include=['float64']).columns
    df_metrics[float_cols] = df_metrics[float_cols].round(4)

    print(f'Computed metrics for {len(df_metrics)} neighbourhoods '
          f'across {df_metrics["city"].nunique()} cities')
    return df_metrics


def compute_persona_scores(df, df_geo, df_poi):
    df = df.copy()
    result_dfs = []

    for city in df['city'].unique():
        city_df = df[df['city'] == city].copy()

        # Join listing density from geo data
        city_geo = df_geo[df_geo['city'] == city][['NEIGHBOURHOOD', 'AREA_SQKM']]
        city_df = city_df.merge(
            city_geo.rename(columns={'NEIGHBOURHOOD': 'neighbourhood_cleansed'}),
            on='neighbourhood_cleansed',
            how='left'
        )
        city_df['listing_density'] = (
            city_df['listing_count'] / city_df['AREA_SQKM']
        )

        # Join POI data
        city_poi = df_poi[['NEIGHBOURHOOD', 'poi_transport',
                            'poi_attractions', 'poi_dining'
                           ]].rename(
            columns={'NEIGHBOURHOOD': 'neighbourhood_cleansed'}
        )
        city_df = city_df.merge(
            city_poi, on='neighbourhood_cleansed', how='left'
        )
        city_df[['poi_transport', 'poi_attractions',
                  'poi_dining']] = city_df[[
            'poi_transport', 'poi_attractions', 'poi_dining'
        ]].fillna(0)

        # Compute weighted POI score per neighbourhood
        city_df['poi_score'] = (
            city_df['poi_transport']   * 0.40 +
            city_df['poi_attractions'] * 0.35 +
            city_df['poi_dining']      * 0.25
        )

        # Combine POI score with listing density — POI 70%, density 30%
        city_df['location_score'] = (
            city_df['poi_score']       * 0.70 +
            city_df['listing_density'] * 0.30
        )

        # Normalise all signals within city
        norm_price_inv  = minmax_norm(city_df['avg_price'], invert=True)
        norm_avg_price  = minmax_norm(city_df['avg_price'])
        norm_occupancy  = minmax_norm(city_df['avg_occupancy'])
        norm_revenue    = minmax_norm(city_df['avg_revenue'])
        norm_rating     = minmax_norm(city_df['avg_review_rating'])
        norm_location   = minmax_norm(city_df['location_score'].fillna(0))

        # Yield Maximiser — revenue and occupancy focused
        city_df['score_yield_maximiser'] = (
            (norm_revenue   * 0.30 +
             norm_occupancy * 0.30 +
             norm_avg_price * 0.20 +
             norm_rating    * 0.10 +
             norm_location  * 0.10) * 100
        ).round(2)

        # Occupancy Optimiser — occupancy and stability focused
        city_df['score_occupancy_optimiser'] = (
            (norm_occupancy  * 0.40 +
             norm_revenue    * 0.20 +
             norm_rating     * 0.20 +
             norm_avg_price  * 0.10 +
             norm_location   * 0.10) * 100
        ).round(2)

        # Quality Host — rating and guest experience focused
        city_df['score_quality_host'] = (
            (norm_rating     * 0.40 +
             norm_occupancy  * 0.20 +
             norm_price_inv  * 0.20 +
             norm_revenue    * 0.10 +
             norm_location   * 0.10) * 100
        ).round(2)

        result_dfs.append(city_df)

    df_out = pd.concat(result_dfs, ignore_index=True)
    df_out['computed_at'] = pd.Timestamp.now()

    print(f'Persona scores computed for {len(df_out)} neighbourhoods')
    return df_out


def write_gold(session, df):
    session.write_pandas(
        df,
        'BOROUGH_SUMMARY',
        database='AIRBNB_INVESTMENT_DB',
        schema='GOLD',
        overwrite=True
    )
    print(f'Written {len(df)} rows to GOLD.BOROUGH_SUMMARY')


def validate(session):
    df_val = session.sql("""
        SELECT
            city,
            COUNT(*) AS neighbourhood_count,
            ROUND(AVG(avg_price), 2) AS mean_avg_price,
            ROUND(AVG(avg_occupancy), 4) AS mean_avg_occupancy,
            ROUND(AVG(location_score), 4) AS avg_location_score,
            ROUND(MIN(score_yield_maximiser), 2) AS min_ym_score,
            ROUND(MAX(score_yield_maximiser), 2) AS max_ym_score,
            ROUND(MIN(score_occupancy_optimiser), 2) AS min_oo_score,
            ROUND(MAX(score_occupancy_optimiser), 2) AS max_oo_score,
            ROUND(MIN(score_quality_host), 2) AS min_qh_score,
            ROUND(MAX(score_quality_host), 2) AS max_qh_score
        FROM AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY
        GROUP BY city
        ORDER BY city
    """).to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
df_silver  = load_silver(session)
df_geo     = load_geo_data(session)
df_poi     = load_poi_data(session)
df_metrics = compute_metrics(df_silver)
df_scored  = compute_persona_scores(df_metrics, df_geo, df_poi)
write_gold(session, df_scored)
validate(session)

In [ ]:
df_val = session.sql("""
    SELECT
        city,
        COUNT(*) AS neighbourhood_count,
        ROUND(AVG(avg_price), 2) AS mean_avg_price,
        ROUND(AVG(avg_occupancy), 4) AS mean_avg_occupancy,
        ROUND(AVG(location_score), 4) AS avg_location_score,
        ROUND(MIN(score_yield_maximiser), 2) AS min_ym_score,
        ROUND(MAX(score_yield_maximiser), 2) AS max_ym_score,
        ROUND(MIN(score_occupancy_optimiser), 2) AS min_oo_score,
        ROUND(MAX(score_occupancy_optimiser), 2) AS max_oo_score,
        ROUND(MIN(score_quality_host), 2) AS min_qh_score,
        ROUND(MAX(score_quality_host), 2) AS max_qh_score
    FROM AIRBNB_INVESTMENT_DB.GOLD.BOROUGH_SUMMARY
    GROUP BY city
    ORDER BY city
""").to_pandas()
print(df_val)

## Borough Summary Complete

`GOLD.BOROUGH_SUMMARY` is ready for Streamlit consumption.
Each row represents one neighbourhood × city with three persona investment scores.

Scores are normalised **per city** (not cross-city), so a score of 80 for Bristol and 80 for
London mean "top of that city's ranking" — they are not directly comparable across cities.

Personas scored:
- **Yield Maximiser** — weighted toward revenue (40%) and occupancy (30%)
- **Occupancy Optimiser** — weighted toward occupancy (50%) and revenue (20%)
- **Quality Host** — weighted toward review rating (50%) and price accessibility (20%)

Source: `AIRBNB_INVESTMENT_DB.SILVER.LISTINGS_CLEANED` — Bristol, London, Greater Manchester.